In [1]:
import glob
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [2]:
# Get gene trait associations
RAP_DIR = 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results'

# ASSOC_FILE = 'loftee_mac20_associations_bh_corrected.parquet'
# ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR.parquet'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'

LOCAL_DIR = '/home/dnanexus/data_dir/'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = (
    pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

# CORR_FILE = 'regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet'
# !dx download {RAP_DIR}/{CORR_FILE} -o {LOCAL_DIR}

# loftee_corrs = (
#     pl.read_parquet(f'{LOCAL_DIR}/{CORR_FILE}')
#     .with_columns(
#         loftee_corr = pl.col('correlation'),
#         loftee_corr_abs = pl.col('correlation').abs(),
#         loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
#     )
#     .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
# )

# gene_trait_df = (
#     gene_trait_df
#     .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
#     .drop_nans()
#     .sort('loftee_corr_abs', descending=True)
#     .unique(subset=["region"], keep="first", maintain_order=True)
# )
gene_trait_df

Error: path "/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_
miss20per.parquet" already exists but -f/--overwrite was not set


region,phenotype,pval_fdr
str,str,f64
"""ENSG00000132855""","""apolipoprotein_a_int""",0.000007
"""ENSG00000052841""","""apolipoprotein_a_int""",0.038502
"""ENSG00000110243""","""apolipoprotein_a_int""",0.003376
"""ENSG00000118137""","""apolipoprotein_a_int""",4.5099e-46
"""ENSG00000173064""","""apolipoprotein_a_int""",0.006545
…,…,…
"""ENSG00000182095""","""forced_expiratory_volume_in_1s…",0.02095
"""ENSG00000164741""","""forced_expiratory_volume_in_1s…",0.036208
"""ENSG00000205189""","""forced_expiratory_volume_in_1s…",0.045581


In [3]:
# EUR unrelated individuals

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/sample_lists/unrelated_cauc_samples_3rd_degree.csv -o /home/dnanexus/data_dir/

unrel_eur_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv')['eid'].cast(pl.Utf8).to_list()
unrel_eur_samples[:5]

Error: path "/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv"
already exists but -f/--overwrite was not set


['1000020', '1000107', '1000161', '1000172', '1000221']

In [4]:
# Download phenotypes: covariates and PRS corrected
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/phenotypes/corrected_cov_PRS_traits_EUR.parquet -o /home/dnanexus/data_dir/

phenos = (
    pl.read_parquet('/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet')
    .rename({'individual':'sample'})
    .filter(pl.col('sample').is_in(unrel_eur_samples))
)

# phenos
long_phenos = (
    phenos
    .unpivot(
        index='sample',
        on=gene_trait_df['phenotype'].unique().to_list(),
        variable_name='phenotype',
        value_name='pheno_value'
    )
    .drop_nulls()
)

print(long_phenos['phenotype'].value_counts(sort=True))
long_phenos

Error: path "/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet"
already exists but -f/--overwrite was not set
shape: (102, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u64    │
╞═════════════════════════════════╪════════╡
│ townsend_deprivation_index_at_… ┆ 378461 │
│ waist_circumference_int         ┆ 378281 │
│ hip_circumference_int           ┆ 378244 │
│ standing_height_int             ┆ 378108 │
│ weight_int                      ┆ 377841 │
│ …                               ┆ …      │
│ phosphate_int                   ┆ 330147 │
│ apolipoprotein_a_int            ┆ 328787 │
│ shbg_int                        ┆ 327605 │
│ testosterone_int                ┆ 327366 │
│ direct_bilirubin_int            ┆ 307355 │
└─────────────────────────────────┴────────┘


sample,phenotype,pheno_value
str,str,f64
"""1000020""","""trunk_fatfree_mass_int""",-0.779708
"""1000107""","""trunk_fatfree_mass_int""",0.203538
"""1000161""","""trunk_fatfree_mass_int""",-0.665781
"""1000172""","""trunk_fatfree_mass_int""",-0.720524
"""1000221""","""trunk_fatfree_mass_int""",0.01757
…,…,…
"""4974782""","""monocyte_count_int""",-0.310986
"""5956310""","""monocyte_count_int""",1.092157
"""4301443""","""monocyte_count_int""",-1.45838


In [6]:
mac = 20

RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"

# ANNO_FILE = "annotations_fillna_ukbgym.parquet"
ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_DIR}/{ANNO_FILE}

id_list = (
    pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}')
    .filter(
        pl.col('region').is_in(gene_trait_df.select('region').unique().to_series()),
        pl.col('mac_ukb')<=mac,
    )
    .select('id')
    .unique()
    .collect()
)

id_list

Error: path
"/home/dnanexus/data_dir//annotations_fillna_ukbgym_with_mane.parquet" already
exists but -f/--overwrite was not set


/tmp/ipykernel_362330/3908164256.py:18: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


id
str
"""chr2:61329401:A:G"""
"""chr9:32456323:A:G"""
"""chr7:154007813:TCTC:T"""
"""chr7:18671991:CTA:C"""
"""chr3:97859813:G:C"""
…
"""chr10:29623745:G:T"""
"""chr2:27654335:T:C"""
"""chr2:151458226:A:ATTTTTTTTTTTT"""


In [6]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
    .filter(
        pl.col('gt')==1,
        pl.col('sample').is_in(unrel_eur_samples),
    )
    .join(
        id_list.lazy(),
        on='id',
        how='semi'
    )

    .collect()
)

long_gt

Error: path "/home/dnanexus/data_dir/gt_long.parquet" already exists but
-f/--overwrite was not set


id,sample,gt
str,str,i8
"""chr10:24578613:TATAAA:T""","""1745607""",1
"""chr10:24578613:TATAAA:T""","""4369337""",1
"""chr10:24578613:TATAAA:T""","""3190490""",1
"""chr10:24578613:TATAAA:T""","""2067640""",1
"""chr10:24578613:TATAAA:T""","""1906931""",1
…,…,…
"""chr9:19312532:A:C""","""4707581""",1
"""chr9:19312532:A:C""","""5902273""",1
"""chr9:19312534:A:C""","""2004709""",1


In [7]:
import math

output_dir = '/home/dnanexus/data_dir/appv_phenos'
!mkdir -p {output_dir}

pheno_list = long_phenos['phenotype'].unique().to_list()
CHUNK_SIZE = 10
num_phenos = len(pheno_list)
num_chunks = math.ceil(num_phenos / CHUNK_SIZE)

# Process in Batches
for i in tqdm(range(0, num_phenos, CHUNK_SIZE)):
    # 1. Define the current batch of genes
    chunk_phenos = pheno_list[i : i + CHUNK_SIZE]

    print(f"Processing chunk starting at index: {i}")
    (
        long_phenos.lazy()
        .filter(pl.col('phenotype').is_in(chunk_phenos))
        .join(
            long_gt.lazy(),
            on='sample',
            how='inner'
        )
        .group_by(['id', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_rank=pl.col('mean_pheno_value')
                .rank(method="max")
                .over('phenotype')
                .cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_ptile=(
                pl.col('mean_pheno_value_rank') / pl.len().over('phenotype')
            ).cast(pl.Float32),
        )

        .sink_parquet(f'{output_dir}/tmp_appv_chunk_{i}.parquet')
    )


  0%|          | 0/11 [00:00<?, ?it/s]

Processing chunk starting at index: 0


  9%|▉         | 1/11 [00:38<06:23, 38.34s/it]

Processing chunk starting at index: 10


 18%|█▊        | 2/11 [01:17<05:47, 38.61s/it]

...

 82%|████████▏ | 9/11 [05:30<01:11, 35.92s/it]

Processing chunk starting at index: 90


 91%|█████████ | 10/11 [06:07<00:36, 36.25s/it]

Processing chunk starting at index: 100


100%|██████████| 11/11 [06:15<00:00, 34.10s/it]


In [5]:
combined_output_file = "/home/dnanexus/data_dir/quant_pheno_loftee_mac20_EURunrelated_miss20per_appv_percentiles.parquet"

# 1. Get list of files manually
files = glob.glob(f'{output_dir}/*.parquet')
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
combined_lazy = pl.concat(lfs, how="vertical_relaxed")

# 4. Stream to disk
print("Streaming to disk...")
combined_lazy.sink_parquet(combined_output_file, engine='streaming')
print("Done.")

Found 11 files.
Streaming to disk...
Done.


In [8]:
# Subset appv to gene-trait associations we use (instead of all vs all)
small_output_file = "/home/dnanexus/data_dir/quant_pheno_loftee_mac20_EURunrelated_miss20per_appv_percentiles_small.parquet"

appv_big = pl.scan_parquet(combined_output_file)
id_region = pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}').select(['id', 'region']).unique()

(
    appv_big
    .join(id_region, on='id', how='inner')
    .join(gene_trait_df.lazy(), on=['region', 'phenotype'], how='inner')
    .sink_parquet(small_output_file, engine='streaming')
)

In [9]:
tmp = pl.scan_parquet(combined_output_file)
tmp.head().collect()

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,i32,f32,f32,f32,f32
"""chr17:29494262:G:A""","""arm_fat_mass_left_int""",4,0.445486,0.604305,1.3936219e7,0.760874
"""chr17:29494750:C:T""","""triglycerides_int""",2,0.346514,1.357225,1.2964637e7,0.7187
"""chr17:29496832:T:C""","""arm_fat_mass_left_int""",3,0.651005,0.823139,1.5344562e7,0.837765
"""chr17:29497501:A:C""","""immature_reticulocyte_fraction…",17,-0.094024,0.805131,7.932687e6,0.439377
"""chr17:29498311:C:T""","""diastolic_blood_pressure_autom…",1,0.22589,null,1.153331e7,0.646081


In [10]:
tmp = pl.scan_parquet(combined_output_file)

tmp.select(['phenotype']).collect()['phenotype'].value_counts(sort=True)

phenotype,count
str,u64
"""townsend_deprivation_index_at_…",18466933
"""waist_circumference_int""",18462973
"""hip_circumference_int""",18462041
"""standing_height_int""",18458733
"""weight_int""",18452506
…,…
"""phosphate_int""",17245087
"""apolipoprotein_a_int""",17208110
"""shbg_int""",17178512


In [11]:
!dx upload {combined_output_file} --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/

[===========================================================>] Uploaded 34,557,013,081 of 34,557,013,081 bytes (100%) /home/dnanexus/data_dir/quant_pheno_loftee_mac20_EURunrelated_miss20per_appv_percentiles.parquet=========================================================>  ] Uploaded 33,520,877,568 of 34,557,013,081 bytes (97%) /home/dnanexus/data_dir/quant_pheno_loftee_mac20_EURunrelated_miss20per_appv_percentiles.parquet                                                         ] Uploaded 1,509,949,440 of 34,557,013,081 bytes (4%) /home/dnanexus/data_dir/quant_pheno_loftee_mac20_EURunrelated_miss20per_appv_percentiles.parquet[========================================>                   ] Uploaded 23,353,884,672 of 34,557,013,081 bytes (68%) /home/dnanexus/data_dir/quant_pheno_loftee_mac20_EURunrelated_miss20per_appv_percentiles.parquet[======================================================>     ] Uploaded 31,910,264,832 of 34,557,013,081 bytes (92%) /home/dnanexus/data_dir/quant_pheno_l